In [38]:
import pandas as pd
import json
from dotenv import load_dotenv
import os
import requests
import gc

In [39]:
load_dotenv()

FOCUS_DROP_IDENTIFIERS = [
    'daily_grind_chance',
    'daily_grind_guaranteed'
]
QUICKPLAY_NAME = 'Quickplay'

# BUNGIE JSON KEYS
CHARACTER_ACTIVITIES_KEY = 'characterActivities'
AVAILABLE_ACTIVITIES_KEY = 'availableActivities'
VISIBLE_REWARDS_KEY = 'visibleRewards'
REWARDS_ITEMS_KEY = 'rewardItems'
UISTYLE_KEY = 'uiStyle'
ITEM_QUANTITY_KEY = 'itemQuantity'
ACTIVITY_HASH_KEY = 'activityHash'
ITEM_HASH_KEY = 'itemHash'
DISPLAY_PROPERTIES_KEY = 'displayProperties'
MATCHMAKING_KEY = 'matchmaking'
MAX_PLAYERS_KEY = 'maxPlayers'
EQUIPPING_BLOCK_KEY = 'equippingBlock'
AMMO_TYPE_KEY = 'ammoType'

# PANDAS KEYS
DATA_KEY = 'data'
NAME_KEY = 'name'
ITEM_NAME_KEY = 'itemName'
ACTIVITY_NAME_KEY = 'activityName'

# Request Keys
RESPONSE_KEY = 'Response'
VERSION_KEY = 'version'

# BUNGIE
API_KEY = os.getenv("BUNGIE_API_KEY")
MEMBERSHIP_ID = os.getenv("MEMBERSHIP_ID")
MEMBERSHIP_TYPE = os.getenv("MEMBERSHIP_TYPE")

REQUEST_HEADERS = {"X-API-Key": API_KEY}
BASE = "https://www.bungie.net"
MANIFEST_URL = "/Platform/Destiny2/Manifest/"
PROFILE_URL = f"/Platform/Destiny2/{MEMBERSHIP_TYPE}/Profile/{MEMBERSHIP_ID}/"

# JSON Paths
ACTIVITY_DEFINITION_FILENAME = "DestinyActivityDefinition.json"
INVENTORY_ITEM_LITE_DEFINITION_FILENAME = "DestinyInventoryItemLiteDefinition.json"
CHARACTER_DEFINITION_FILENAME = "Character.json"

# Local Save Files
MANIFEST_FILENAME = "Manifest.json"
ACTIVITY_HASHES_FILENAME = "ActivityHashes.pkl"
ITEM_HASHES_FILENAME = "ItemHashes.pkl"
CHARACTER_DATA_FILENAME = "CharacterData.pkl"
ACTIVITY_ITEM_NAMES_FILENAME = "Names.pkl"


In [40]:
## EXCEPTIONS ##
class FocusError(Exception):
    def __init__(self, message):
        super().__init__(message)

class FocusKeyError(FocusError):
    def __init__(self, key, message='Key not found'):
        self.key = key
        super().__init__(f'{message}: given key {key} was not found.')
        
class FocusNoValuesFoundError(FocusError):
    def __init__(self, message='No values found'):
        super().__init__(f'{message}')

class FocusHashCountError(FocusError):
    def __init__(self, activity_count, item_count, message='Item and activity count not equal'):
        super().__init__(f'{message}: {activity_count} activities and {item_count} items were found')

class FocusRequestError(FocusError):
    def __init__(self, message='Request failed'):
        super().__init__(f'{message}')

class FocusParseError(FocusError):
    def __init__(self, message='Parse failed'):
        super().__init__(f'{message}')


In [41]:
def get_json_from_filename(filename):
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError as e:
        print(f'FileNotFoundError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except IsADirectoryError as e:
        print(f'IsADirectoryError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except PermissionError as e:
        print(f'PermissionError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except TimeoutError as e:
        print(f'TimeoutError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except UnicodeDecodeError as e:
        print(f'TimeoutError when getting data from file: {e}')


In [42]:
def save_json_with_filename(filename, data):
    try:
        with open(filename, 'w') as f:
            json.dump(data, f)
    except IsADirectoryError as e:
        print(f'IsADirectoryError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except PermissionError as e:
        print(f'PermissionError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except TimeoutError as e:
        print(f'TimeoutError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except UnicodeDecodeError as e:
        print(f'TimeoutError when getting data from file: {e}')


In [43]:
def get_activities_dict(activity_data):
    """
    Raises: FocusKeyError, FocusNoValuesError
    """
    # get teh data for characters
    activity_found_data = activity_data.get(CHARACTER_ACTIVITIES_KEY)
    if activity_found_data is None:
        raise FocusKeyError(CHARACTER_ACTIVITIES_KEY, 'Key not found during get_activities_dict()')

    activity_found_data = activity_found_data.get(DATA_KEY)
    if activity_found_data is None:
        raise FocusKeyError(DATA_KEY, 'Key not found during get_activities_dict()')

    activity_found_data = activity_found_data.values()
    if (activity_found_data is None) or (len(activity_found_data) == 0):
        raise FocusNoValuesFoundError('No activities found during get_activities_dict()')

    # grab the data from teh first character (all characters have the same Focus Drops,
    # barring class distinctions (i.e. Warlock Bond, Titan Helm, etc)
    # therefore, we don't need a specific character ID, which could present
    # problems in the future.
    activity_found_data = next(iter(activity_found_data))
    if activity_found_data is not None:
        activity_found_data = activity_found_data.get(AVAILABLE_ACTIVITIES_KEY)
        if activity_found_data is not None:
            # TODO: REMOVE
            with open('characterActivities.json', 'w') as f:
                json.dump(activity_found_data, f)
            return activity_found_data
        else:
            raise FocusKeyError(AVAILABLE_ACTIVITIES_KEY, 'Key not found during get_activities_dict()')
    else:
        raise FocusNoValuesFoundError('No activities found during get_activities_dict()')



In [44]:
def activity_df_from_data(activity_data):
    """
    Raises: FocusNoValuesFoundError
    """
    activities_df = pd.DataFrame(activity_data)
    if activities_df.empty or activities_df is None:
        raise FocusNoValuesFoundError('Unable to build DataFrame from activity data in activity_data_to_df()')
    # explode 'visibleRewards' lists into rows
    activities_df = activities_df.explode(VISIBLE_REWARDS_KEY)
    if activities_df is None or activities_df.empty or activities_df.size == 0:
        raise FocusNoValuesFoundError('Exploded DataFrame in activity_data_to_df() is empty')
    return activities_df

In [45]:
########################################
# DataFrame Helper Functions
########################################

def extract_reward_items(value):
    value = value.iloc[0]
    if pd.notna(value) is True:
        if value is not None:
            return value.get(REWARDS_ITEMS_KEY)
        else:
            return None
    else:
        return None

def extract_uistyle_validity(value):
    value = value.get(REWARDS_ITEMS_KEY)
    if value is not None:
        uistyle = value.get(UISTYLE_KEY)
        if uistyle is not None:
            if uistyle in FOCUS_DROP_IDENTIFIERS:
                return True
            else:
                return False
        else:
            return False
    else:
        return False

def extract_item_hash(value):
    value = value.iloc[0]
    if value is None:
        return None
    value = value.get(ITEM_QUANTITY_KEY)
    if value is None:
        return None
    value = value.get(ITEM_HASH_KEY)
    if value is None:
        return None
    else:
        return value

In [46]:
def rewards_df_from_activity_df(exploded_activity_df):
    """
    Raises: FocusNoValuesFoundError
    """
    # extract series from visibleRewards
    reward_series = exploded_activity_df.get([VISIBLE_REWARDS_KEY]).apply(extract_reward_items, axis=1)

    if reward_series is None or reward_series.empty or reward_series.size == 0:
        raise FocusNoValuesFoundError('No rewards found during rewards_df_from_activity_df()')

    # update dataframe with new series appended and explode rewardItems
    rewards_exploded_df = exploded_activity_df.assign(rewardItems=reward_series).explode(REWARDS_ITEMS_KEY)
    return rewards_exploded_df

In [47]:
def focus_items_df_from_rewards_df(rewards_exploded_df):
    """
    Raises: FocusNoValuesFoundError
    """
    # extract uiStyle validity (is a focus drop?)
    uistyle_extracted = rewards_exploded_df.apply(extract_uistyle_validity, axis=1)
    # mask dataframe with new series
    df = rewards_exploded_df[uistyle_extracted]
    if df is None or df.empty or df.size == 0:
        raise FocusNoValuesFoundError('No items found during focus_items_df_from_rewards_df()')
    return df

In [48]:
def extract_activity_item_hashes(filtered_df):
    """
    Raises: FocusNoValuesFoundError, FocusHashCountError
    """
    item_hashes = filtered_df.get([REWARDS_ITEMS_KEY]).apply(extract_item_hash, axis=1)

    if item_hashes is None or item_hashes.empty or item_hashes.size == 0:
        raise FocusNoValuesFoundError('No item hashes found during extract_activity_item_hashes()')
    activity_hashes = filtered_df.get(ACTIVITY_HASH_KEY)
    if activity_hashes is None or activity_hashes.empty or activity_hashes.size == 0:
        raise FocusNoValuesFoundError('No hashes found during extract_activity_item_hashes()')
    if item_hashes.size != activity_hashes.size:
        raise FocusHashCountError(activity_hashes.size, item_hashes.size, 'Unequal hash counts in extract_activity_item_hashes()')

    # Remove duplicate activities and merge the hashes together in a list of dictionaries
    return merge_activity_item_hashes(activity_hashes, item_hashes)


In [49]:
# Remove duplicate activities and merge the hashes together in a list of dictionaries
def merge_activity_item_hashes(activity_hashes, item_hashes):

    activity_hashes_list = []
    item_hashes_list = []
    for i in range(len(activity_hashes)):
        item_hash = int(item_hashes.iloc[i])
        if item_hash not in item_hashes_list:
            item_hashes_list.append(item_hash)
            activity_hashes_list.append(int(activity_hashes.iloc[i]))
    df = pd.DataFrame().assign(activityHash=activity_hashes_list, itemHash=item_hashes_list)
    return df

In [50]:
# Remove duplicate activities and merge the hashes together in a list of dictionaries
def get_remote_manifest(request_url):
    """
    Raises: FocusRequestError
    """
    response = requests.get(request_url, headers=REQUEST_HEADERS, timeout=10)
    if response is None:
        raise FocusRequestError('Error receiving Bungie API response during manifest request')
    try:
        response.raise_for_status()
        remote_manifest = response.json()
    except requests.exceptions.HTTPError as e:
        raise FocusRequestError(f'Error: Response: {e.response.text} | Request: {e.request.url} | {e.strerror} | {e.errno}')
    except json.decoder.JSONDecodeError as e:
        raise  FocusRequestError('Error decoding JSON from remote manifest in get_remote_manifest()')
    if remote_manifest is None:
        raise FocusRequestError('Error: no JSON from remote manifest in get_remote_manifest()')
    return remote_manifest

In [51]:
# Remove duplicate activities and merge the hashes together in a list of dictionaries
def get_local_manifest():
    local_manifest = None
    if os.path.exists(MANIFEST_FILENAME):
        local_manifest = get_json_from_filename(MANIFEST_FILENAME)

    return local_manifest

In [52]:
# Remove duplicate activities and merge the hashes together in a list of dictionaries
def is_manifest_up_to_date(remote_manifest, local_manifest):
    """
    Raises: FocusRequestError
    """
    local_response_lhs = local_manifest.get(RESPONSE_KEY)
    if local_response_lhs is None:
        raise FocusRequestError('Error getting response from local manifest')
    local_version_lhs = local_response_lhs.get(VERSION_KEY)
    if local_version_lhs is None:
        raise FocusRequestError('Error getting version from local manifest')

    remote_response_rhs = remote_manifest.get(RESPONSE_KEY)
    if remote_response_rhs is None:
        raise FocusRequestError('Error getting response from remote manifest')
    remote_version_rhs = remote_response_rhs.get(VERSION_KEY)
    if remote_version_rhs is None:
        raise FocusRequestError('Error getting version from remote manifest')

    # this checks the need to update the manifest
    if local_version_lhs != remote_version_rhs:
        print(f"Local manifest version diff from remote. Updating manifest")
        print(f"Local version: {local_version_lhs}")
        print(f"Remote version: {remote_version_rhs}")
        return False
    else:
        return True

In [53]:
def request_and_update_manifest(args):

    request_url = ""
    for arg in args:
        request_url += arg

    up_to_date = False
    try:
        remote_manifest = get_remote_manifest(request_url)
        local_manifest = get_local_manifest()

        if local_manifest is not None:
            up_to_date = is_manifest_up_to_date(remote_manifest, local_manifest)

        if not up_to_date:
            save_json_with_filename(MANIFEST_FILENAME, remote_manifest)

    except requests.exceptions.Timeout:
        raise RuntimeError("⏳ Bungie API timed out while fetching manifest metadata.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"❌ Bungie API returned HTTP error: {e}")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"⚠️ Network error while fetching manifest metadata: {e}")
    except (json.JSONDecodeError, KeyError) as e:
        raise RuntimeError(f"⚠️ Could not parse manifest metadata: {e}")
    except FocusRequestError as e:
        print(f'Error during manifest requests: {e}')

    gc.collect()
    return up_to_date

In [54]:
def save_json_from_request(request_url, filename):
    try:
        response = requests.get(request_url, headers=REQUEST_HEADERS, timeout=10)
        response.raise_for_status()
        hashes = response.json()
        with open(filename, 'w') as f:
            json.dump(hashes, f)
    except requests.exceptions.Timeout:
        raise RuntimeError("⏳ Bungie API timed out while fetching activity metadata.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"❌ Bungie API returned HTTP error: {e}")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"⚠️ Network error while fetching activity metadata: {e}")
    except (json.JSONDecodeError, KeyError) as e:
        raise RuntimeError(f"⚠️ Could not parse activity metadata: {e}")

In [55]:
def request_activity_hashes(bng_manifest, args):
    """
    Raises: RuntimeError
    """
    request_url = ""
    # request_url_activity_type = ""
    # request_url_activity_mode = ""

    for arg in args:
        request_url += arg
        # request_url_activity_mode += arg
        # request_url_activity_type += arg

    bng_manifest_response = bng_manifest.get(RESPONSE_KEY)
    if bng_manifest_response is None:
        raise FocusRequestError('Error receiving Bungie API response during activity_hashes request')

    request_url += bng_manifest["Response"]["jsonWorldComponentContentPaths"]["en"]["DestinyActivityDefinition"]
    # request_url_activity_mode += bng_manifest["Response"]["jsonWorldComponentContentPaths"]["en"]["DestinyActivityModeDefinition"]
    # request_url_activity_type += bng_manifest["Response"]["jsonWorldComponentContentPaths"]["en"]["DestinyActivityTypeDefinition"]

    try:
        response = requests.get(request_url, headers=REQUEST_HEADERS, timeout=10)
        response.raise_for_status()
        hashes = response.json()

        #TODO: REMOVE FROM PROGRAM
        # save_json_from_request(request_url_activity_type, 'activityTypeDefinition.json')
        # save_json_from_request(request_url_activity_mode, 'activityModeDefinition.json')
    except requests.exceptions.Timeout:
        raise RuntimeError("⏳ Bungie API timed out while fetching activity metadata.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"❌ Bungie API returned HTTP error: {e}")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"⚠️ Network error while fetching activity metadata: {e}")
    except (json.JSONDecodeError, KeyError) as e:
        raise RuntimeError(f"⚠️ Could not parse activity metadata: {e}")
    return hashes

In [56]:
def request_item_hashes(bng_manifest, args):
    """
    Raises: RuntimeError
    """
    request_url = ""
    for arg in args:
        request_url += arg

    request_url += bng_manifest["Response"]["jsonWorldComponentContentPaths"]["en"]["DestinyInventoryItemDefinition"]
    try:
        response = requests.get(request_url, headers=REQUEST_HEADERS)
        response.raise_for_status()
        hashes = response.json()
    except requests.exceptions.Timeout:
        raise RuntimeError("⏳ Bungie API timed out while fetching item metadata.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"❌ Bungie API returned HTTP error: {e}")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"⚠️ Network error while fetching item metadata: {e}")
    except (json.JSONDecodeError, KeyError) as e:
        raise RuntimeError(f"⚠️ Could not parse item metadata: {e}")
    return hashes

In [57]:
def get_character_data():
    """
    Raises: RuntimeError, FocusRequestError
    """
    params = {
        "components": 204  # CharacterActivities
    }
    try:
        response = requests.get(BASE+PROFILE_URL, headers=REQUEST_HEADERS, params=params)
        #TODO: make safer
        if response is None:
            raise FocusRequestError('Error getting character data response during get_character_data()')
        if response.status_code == 200:
            character_data = response.json()["Response"]
            if character_data is None:
                raise FocusRequestError('Error getting character data during get_character_data()')
        else:
            print("Error:", response.status_code, response.text)
            raise FocusRequestError('Response error code during get_character_data()')
    except requests.exceptions.Timeout:
        raise RuntimeError("⏳ Bungie API timed out while fetching profile metadata.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"❌ Bungie API returned HTTP error: {e}")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"⚠️ Network error while fetching profile metadata: {e}")
    except (json.JSONDecodeError, KeyError) as e:
        raise RuntimeError(f"⚠️ Could not parse profile metadata: {e}")
    return character_data

In [58]:
def extract_name_from_display_properties(value):
    return value.iloc[0].get(NAME_KEY)


In [59]:
def extract_matchmaking(value):
    try:
        players = value.get(MATCHMAKING_KEY).get(MAX_PLAYERS_KEY)
    except AttributeError:
        #print(f'No matchmaking data available for:\n{value}')
        players = 0
    return players


In [60]:
def extract_ammo_type(value):
    try:
        ammo_type = value.get(AMMO_TYPE_KEY)
    except AttributeError:
        #print(f'No matchmaking data available for:\n{value}')
        ammo_type = 0
    return ammo_type


In [61]:
def parse_activity_hashes_to_names(activity_json):
    """
    Raises: FocusParseError
    """
    # parse hashes into activity names
    activity_df = pd.DataFrame(activity_json).T

    if activity_df is None:
        raise FocusParseError('Error creating DataFrame during parse_activity_hashes_to_names()')
    try:
        max_players = activity_df.get([MATCHMAKING_KEY]).apply(extract_matchmaking, axis=1)
        activity_df = activity_df.assign(maxPlayers=max_players)
        activity_df = activity_df.assign(name=activity_df.get([DISPLAY_PROPERTIES_KEY]).apply(extract_name_from_display_properties, axis=1)).get([NAME_KEY, MAX_PLAYERS_KEY])
        # update local file
        activity_df.to_pickle(ACTIVITY_HASHES_FILENAME)
    except KeyError:
        raise FocusParseError('Error extracting names during parse_activity_hashes_to_names()')



In [62]:
def parse_item_hashes_to_names(item_json):
    """
    Raises: FocusParseError
    """
    # parse item hashes into item names
    item_df = pd.DataFrame(item_json).T
    if item_df is None:
        raise FocusParseError('Error creating DataFrame during parse_item_hashes_to_names()')
    try:
        ammo_type = item_df.get(EQUIPPING_BLOCK_KEY).apply(extract_ammo_type)
        item_df = item_df.assign(ammoType=ammo_type)
        item_df = item_df.assign(name=item_df.get([DISPLAY_PROPERTIES_KEY]).apply(extract_name_from_display_properties, axis=1)).get([NAME_KEY, AMMO_TYPE_KEY, 'itemTypeDisplayName'])
        # update local file
        item_df.to_pickle(ITEM_HASHES_FILENAME)
    except KeyError:
        raise FocusParseError('Error extracting names during parse_item_hashes_to_names()')

In [63]:
def parse_character_data_to_activities_items(character_data):
    """
    Raises: FocusParseError
    """
    # parse character bonus focus drops into activity and item hashes
    try:
        character_data = get_activities_dict(character_data)
        character_data = activity_df_from_data(character_data)
        character_data = rewards_df_from_activity_df(character_data)
        character_data = focus_items_df_from_rewards_df(character_data)
        character_df = extract_activity_item_hashes(character_data)
        character_df.to_pickle(CHARACTER_DATA_FILENAME)

    except FocusKeyError or FocusNoValuesFoundError or FocusHashCountError as e:
        raise FocusParseError(f'Error extracting activity and item hashes from character_data: {e}')
    except KeyError as e:
        raise FocusParseError(f'Error parsing character data: {e}')

In [64]:
def get_remote_json():
    """
    Raises: FocusRequestError, FocusParseError
    """
    manifest, activities_hashes, items_hashes, character_data = None, None, None, None

    manifest = get_local_manifest()
    if manifest is None:
        raise FocusRequestError('Error getting local manifest during get_remote_json()')

    try:
        activities_hashes = request_activity_hashes(manifest, BASE)
        items_hashes = request_item_hashes(manifest, BASE)
        character_data = get_character_data()

        parse_activity_hashes_to_names(activities_hashes)
        parse_item_hashes_to_names(items_hashes)
        parse_character_data_to_activities_items(character_data)
    except FocusRequestError or FocusParseError as e:
        raise e

In [65]:
def get_dataframes_from_local():
    activity_df, item_df, character_df = None, None, None
    done = False
    while not done:
        if os.path.exists(ACTIVITY_HASHES_FILENAME):
            activity_df = pd.read_pickle(ACTIVITY_HASHES_FILENAME)

        if os.path.exists(ITEM_HASHES_FILENAME):
            item_df = pd.read_pickle(ITEM_HASHES_FILENAME)

        if os.path.exists(CHARACTER_DATA_FILENAME):
            character_df = pd.read_pickle(CHARACTER_DATA_FILENAME)

        if activity_df is None or item_df is None or character_df is None:
            try:
                get_remote_json()
            except (FocusRequestError, FocusParseError) as e:
                print(e)
        else:
            done = True

    return activity_df, item_df, character_df

In [66]:
def get_name_from_df(activity_hash, item_hash, activity_df, item_df):
    """
    Raises: FocusParseError
    """
    activity_name, item_name = None, None
    try:
        display(item_df)
        activity = activity_df.loc[f'{activity_hash}']
        item = item_df.loc[f'{item_hash}']
        if activity is not None and not activity.empty and item is not None and not item.empty:
            activity_name = activity.loc[NAME_KEY].split(":", 1)[0]
            if QUICKPLAY_NAME in activity_name:
                activity_max_players = activity.loc[MAX_PLAYERS_KEY]
                if activity_max_players == 1:
                    activity_name += ' (Solo Ops)'
                elif activity_max_players == 3:
                    activity_name += ' (Fireteam Ops)'
                else:
                    activity_name += ' (Arena Ops)'
            item_name = item.loc[NAME_KEY]
            ammo_type = item.loc[AMMO_TYPE_KEY]
            if ammo_type == 0:
                #TODO
                pass
    except KeyError as e:
        raise FocusParseError(f'Error parsing activity or item name in get_name_from_df(): {e}')
    return activity_name, item_name

In [67]:
# Remove duplicate activities and merge the hashes together in a list of dictionaries
def map_hash_df_to_names(activity_df, item_df, character_df):
    """
    Raises: FocusParseError
    """
    if activity_df is None or activity_df.empty or item_df is None or item_df.empty or character_df is None or character_df.empty:
        raise FocusParseError('Empty dataframe(s) during map_hash_df_to_names()')
    activities = character_df.get(ACTIVITY_HASH_KEY)
    items = character_df.get(ITEM_HASH_KEY)
    if activities is None or items is None:
        raise FocusParseError('Error parsing activities and items during map_hash_df_to_names(): no activities or items')
    activity_item_df = pd.DataFrame(columns=[ACTIVITY_NAME_KEY, ITEM_NAME_KEY])
    try:
        for activity, item in zip(activities, items):
            activity_name, item_name = get_name_from_df(activity, item, activity_df, item_df)
            new_row = pd.DataFrame({ACTIVITY_NAME_KEY : [activity_name], ITEM_NAME_KEY : [item_name]})
            activity_item_df = pd.concat([activity_item_df, new_row], ignore_index=True)
    except KeyError or TypeError as e:
        raise FocusParseError(f'Error getting item/activity names from hashes in map_hash_df_to_names(): {e}')
    except FocusParseError as e:
        raise e

    activity_item_df.to_pickle(ACTIVITY_ITEM_NAMES_FILENAME)


In [68]:
# Remove duplicate activities and merge the hashes together in a list of dictionaries
def update_local_from_remote():
    """
    Raises: FocusParseError, FocusRequestError
    """
    try:
        get_remote_json()
        activity_df, items_df, character_df = get_dataframes_from_local()
        map_hash_df_to_names(activity_df, items_df, character_df)
    except (FocusParseError, FocusRequestError) as e:
        raise e

In [69]:
# Remove duplicate activities and merge the hashes together in a list of dictionaries
def manifest_scheduler(args):
    up_to_date = request_and_update_manifest(args)

    if not up_to_date or not os.path.exists(ACTIVITY_HASHES_FILENAME) or not os.path.exists(ACTIVITY_ITEM_NAMES_FILENAME):
        try:
            update_local_from_remote()
        except (FocusParseError, FocusRequestError) as e:
            print(e)


In [70]:
# Remove duplicate activities and merge the hashes together in a list of dictionaries
def get_names_from_local():
    return pd.read_pickle(ACTIVITY_ITEM_NAMES_FILENAME)

In [71]:
def program_start():

    '''
    1. check if manifest changed
        a. if true:
            i. request info
                A. update local activity definition file
                    1. parse into activity names & hashes
                B. update local item definition file
                    2. parse into item names & hashes
                C. update local character data file
                    3. parse into item activity hashes
    2. pull data from local files
        a. load activity names & hashes
        b. load item names & hashes
        c. load character activity & item hashes
        d. map character hashes to names
            i.
    '''

    #######################################
    # TODO: EVERY DAY / FEW HOURS
    #######################################
    manifest_scheduler([BASE, MANIFEST_URL])

    #######################################
    # TODO: UPON REQUEST
    #######################################
    names_df = get_names_from_local()

    str_builder = ''
    for i in names_df.index:
        str_builder += names_df.iloc[i].get(ACTIVITY_NAME_KEY) + ' : ' + names_df.iloc[i].get(ITEM_NAME_KEY) + '\n'

    print(str_builder)




In [72]:
program_start()

,name,ammoType,itemTypeDisplayName
2899766705,Gauntlets,0,Gauntlets
648507367,Chest Armor,0,Chest Armor
1364005110,Hunter Cloak,0,Hunter Cloak
997252576,Helmet,0,Helmet
2731019523,Leg Armor,0,Leg Armor
...,...,...,...
456184898,The Tenets,0,Quest Step
1303879934,The Tenets,0,Quest Step
1844297021,The Tenets,0,Quest Step
2125848607,Xenology,0,Quest Step


,name,ammoType,itemTypeDisplayName
2899766705,Gauntlets,0,Gauntlets
648507367,Chest Armor,0,Chest Armor
1364005110,Hunter Cloak,0,Hunter Cloak
997252576,Helmet,0,Helmet
2731019523,Leg Armor,0,Leg Armor
...,...,...,...
456184898,The Tenets,0,Quest Step
1303879934,The Tenets,0,Quest Step
1844297021,The Tenets,0,Quest Step
2125848607,Xenology,0,Quest Step


,name,ammoType,itemTypeDisplayName
2899766705,Gauntlets,0,Gauntlets
648507367,Chest Armor,0,Chest Armor
1364005110,Hunter Cloak,0,Hunter Cloak
997252576,Helmet,0,Helmet
2731019523,Leg Armor,0,Leg Armor
...,...,...,...
456184898,The Tenets,0,Quest Step
1303879934,The Tenets,0,Quest Step
1844297021,The Tenets,0,Quest Step
2125848607,Xenology,0,Quest Step


,name,ammoType,itemTypeDisplayName
2899766705,Gauntlets,0,Gauntlets
648507367,Chest Armor,0,Chest Armor
1364005110,Hunter Cloak,0,Hunter Cloak
997252576,Helmet,0,Helmet
2731019523,Leg Armor,0,Leg Armor
...,...,...,...
456184898,The Tenets,0,Quest Step
1303879934,The Tenets,0,Quest Step
1844297021,The Tenets,0,Quest Step
2125848607,Xenology,0,Quest Step


,name,ammoType,itemTypeDisplayName
2899766705,Gauntlets,0,Gauntlets
648507367,Chest Armor,0,Chest Armor
1364005110,Hunter Cloak,0,Hunter Cloak
997252576,Helmet,0,Helmet
2731019523,Leg Armor,0,Leg Armor
...,...,...,...
456184898,The Tenets,0,Quest Step
1303879934,The Tenets,0,Quest Step
1844297021,The Tenets,0,Quest Step
2125848607,Xenology,0,Quest Step


...

,name,ammoType,itemTypeDisplayName
2899766705,Gauntlets,0,Gauntlets
648507367,Chest Armor,0,Chest Armor
1364005110,Hunter Cloak,0,Hunter Cloak
997252576,Helmet,0,Helmet
2731019523,Leg Armor,0,Leg Armor
...,...,...,...
456184898,The Tenets,0,Quest Step
1303879934,The Tenets,0,Quest Step
1844297021,The Tenets,0,Quest Step
2125848607,Xenology,0,Quest Step


,name,ammoType,itemTypeDisplayName
2899766705,Gauntlets,0,Gauntlets
648507367,Chest Armor,0,Chest Armor
1364005110,Hunter Cloak,0,Hunter Cloak
997252576,Helmet,0,Helmet
2731019523,Leg Armor,0,Leg Armor
...,...,...,...
456184898,The Tenets,0,Quest Step
1303879934,The Tenets,0,Quest Step
1844297021,The Tenets,0,Quest Step
2125848607,Xenology,0,Quest Step


,name,ammoType,itemTypeDisplayName
2899766705,Gauntlets,0,Gauntlets
648507367,Chest Armor,0,Chest Armor
1364005110,Hunter Cloak,0,Hunter Cloak
997252576,Helmet,0,Helmet
2731019523,Leg Armor,0,Leg Armor
...,...,...,...
456184898,The Tenets,0,Quest Step
1303879934,The Tenets,0,Quest Step
1844297021,The Tenets,0,Quest Step
2125848607,Xenology,0,Quest Step


,name,ammoType,itemTypeDisplayName
2899766705,Gauntlets,0,Gauntlets
648507367,Chest Armor,0,Chest Armor
1364005110,Hunter Cloak,0,Hunter Cloak
997252576,Helmet,0,Helmet
2731019523,Leg Armor,0,Leg Armor
...,...,...,...
456184898,The Tenets,0,Quest Step
1303879934,The Tenets,0,Quest Step
1844297021,The Tenets,0,Quest Step
2125848607,Xenology,0,Quest Step


Sparrow Racing League : Vikala SMG4
Quickplay (Fireteam Ops) : Service Revolver
The Dark Priestess : Convened Recurve
The Devils' Lair : Luminopotent Helm
Savathûn's Spire : Main Ingredient
The Coil : Lotus-Eater
Midtown : Whatchamacallit
Quickplay (Arena Ops) : Bellowing Giant
Astral Alignment : Hammerhead
Quickplay (Solo Ops) : Resounding
Skywatch : Chain of Command
Mayhem : Joxer's Longsword
Gambit : Python
Warden of Nothing : D.F.A.

